In [1]:
import json  # Parse BigQuery's JSON response into Python records.
from io import StringIO  # Let pandas read BigQuery command output held in memory.
import os  # Set a temporary Matplotlib configuration directory below.
import subprocess  # Run authenticated BigQuery CLI commands from Python.
import tempfile  # Find a safe temporary directory for Matplotlib files.
from pathlib import Path  # Build filesystem paths in an OS-independent way.

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "amazon_books_matplotlib"),
)

import matplotlib.pyplot as plt  # Create plots later in the notebook.
import numpy as np  # Numerical arrays and calculations.
import pandas as pd  # Load, inspect, and transform table-shaped data.
from IPython.display import display

PROJECT = "wagon-bootcamp-501605"
DATASET = "amazon_books_2023"
TABLE = "business_money_usable_tocs_9034"
LOCATION = "asia-northeast1"
RANDOM_STATE = 42
FULL_TABLE = f"{PROJECT}.{DATASET}.{TABLE}"
print(FULL_TABLE)

wagon-bootcamp-501605.amazon_books_2023.business_money_usable_tocs_9034


# Prepare the usable TOCs

The BigQuery table is already a curated source: every record has an exact-ISBN Open Library match and at least three extracted TOC entries. We keep the raw fields unchanged and build a separate, model-ready table.

Following the data-preparation lecture, we audit duplicates and missing data before transforming text. The unit of analysis is `parent_asin`, not ISBN: 39 canonical ISBNs occur in more than one Amazon record.

In [2]:
# Load every source column first. JSON preserves nested and repeated fields
# that BigQuery cannot serialize in CSV.
query = f"""
SELECT *
FROM `{FULL_TABLE}`
ORDER BY parent_asin
"""

command = [
    "bq",
    f"--location={LOCATION}",
    "query",
    "--use_legacy_sql=false",
    "--format=json",
    "--max_rows=10000",
    query,
]
print("1/7 — Full-table BigQuery command is ready.")

1/7 — Full-table BigQuery command is ready.


In [3]:
# Run the full-table query. Its JSON output can preserve nested columns. (50 seconds)
result = subprocess.run(command, capture_output=True, text=True, check=True)
print(f"2/7 — Downloaded {len(result.stdout):,} characters of JSON.")

2/7 — Downloaded 216,668,245 characters of JSON.


In [4]:
# Convert the JSON response into one pandas row per book. (9,034 rows × 38 columns.)
source_books = pd.DataFrame(json.loads(result.stdout))
print(f"3/7 — Created source_books: {source_books.shape[0]:,} rows × {source_books.shape[1]:,} columns.")

3/7 — Created source_books: 9,034 rows × 38 columns.


In [5]:
# Store identifier-like values as text so leading zeros and ISBN-10's final X are never lost.
identifier_columns = [
    "parent_asin", "amazon_isbn_10", "amazon_isbn_13",
    "canonical_isbn_13", "ol_edition_key",
]
for column in identifier_columns:
    source_books[column] = source_books[column].astype("string")
print("4/7 — Identifier dtypes:")
print(source_books[identifier_columns].dtypes)

4/7 — Identifier dtypes:
parent_asin          string[python]
amazon_isbn_10       string[python]
amazon_isbn_13       string[python]
canonical_isbn_13    string[python]
ol_edition_key       string[python]
dtype: object


In [6]:
# CHECKPOINT — before 5/7: we have preserved the original book-level table.
print("Before TOC flattening")
print(f"• source_books contains {len(source_books):,} books and {len(source_books.columns):,} original columns.")
print("• Each row is one Amazon book; nested fields such as toc_entries are still intact.")
print("• Next we create a separate entry-level view only to validate and clean TOC text.")
display(source_books[["parent_asin", "title", "toc_entry_count", "toc_entries"]].head(3))

Before TOC flattening
• source_books contains 9,034 books and 38 original columns.
• Each row is one Amazon book; nested fields such as toc_entries are still intact.
• Next we create a separate entry-level view only to validate and clean TOC text.


,parent_asin,title,toc_entry_count,toc_entries
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",3,"[{'level': None, 'page': None, 'sequence': '1'..."
1,0007519532,Will there be Donuts?: Better Business One Mee...,5,"[{'level': '0', 'page': None, 'sequence': '1',..."
2,0021057117,California Mathematics Grade 4 (Student Editio...,20,"[{'level': '0', 'page': None, 'sequence': '1',..."


In [7]:
# source_books has one row per book and preserves every original column.
# This separate query does not append to or replace source_books. It creates
# toc_items: one row per nested TOC entry, which lets us check entry order,
# blanks, and duplicates before joining only cleaned toc_text back to books.
#
# Before flattening, one book stores its TOC as a list inside one cell:
# parent_asin = "000216132X"
# toc_entries = [{"sequence": 1, "text": "v. 1. The structures..."},
#                {"sequence": 2, "text": "v. 2. The wheels..."},
#                {"sequence": 3, "text": "v. 3. The perspective..."}]
# After flattening, those become three rows with parent_asin, toc_sequence,
# and toc_text_raw. The displays below use a real book from this table.
example_book = source_books.iloc[0]
example_parent_asin = example_book["parent_asin"]
print(f"Nested TOC — real book: {example_parent_asin} | {example_book['title']}")
display(pd.DataFrame(example_book["toc_entries"])[["sequence", "text"]])
toc_query = f"""
SELECT parent_asin, raw_toc_item_count, toc_entry_count,
  entry.sequence AS toc_sequence, entry.text AS toc_text_raw,
  entry.level AS toc_level, entry.page AS toc_page
FROM `{FULL_TABLE}`
CROSS JOIN UNNEST(toc_entries) AS entry
ORDER BY parent_asin, toc_sequence
"""
toc_command = [
    "bq", f"--location={LOCATION}", "query",
    "--use_legacy_sql=false", "--format=csv", "--max_rows=1000000", toc_query,
]
print("5/7 — Flattened TOC query is ready.")

Nested TOC — real book: 000216132X | The wheels of commerce, vol.2: civilisation and capitalism 15th-18th


,sequence,text
0,1,v. 1. The structures of everyday life : the li...
1,2,v. 2. The wheels of commerce
2,3,v. 3. The perspective of the world.


5/7 — Flattened TOC query is ready.


In [8]:
# Run the simple flat query and read its CSV response into pandas.
toc_result = subprocess.run(toc_command, capture_output=True, text=True, check=True)
toc_items = pd.read_csv(StringIO(toc_result.stdout), dtype={"parent_asin": "string"})
print(f"6/7 — Loaded {len(toc_items):,} TOC entries for {toc_items['parent_asin'].nunique():,} books.")

6/7 — Loaded 141,495 TOC entries for 9,034 books.


In [9]:
# Inspect one book without truncating its title or TOC text.
BOOK_TO_INSPECT = "000216132X"
book_record = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
book_toc = toc_items.loc[
    toc_items["parent_asin"] == BOOK_TO_INSPECT,
    ["toc_sequence", "toc_text_raw", "toc_level", "toc_page"],
].sort_values("toc_sequence")

print(f"Amazon title: {book_record['title']}")
print(f"Open Library edition: {book_record['ol_edition_key']}")
print(f"Declared TOC entries: {book_record['toc_entry_count']}")
print("\nFull table of contents:")
for entry in book_toc.itertuples(index=False):
    print(f"{entry.toc_sequence}. {entry.toc_text_raw}")

Amazon title: The wheels of commerce, vol.2: civilisation and capitalism 15th-18th
Open Library edition: /books/OL22124972M
Declared TOC entries: 3

Full table of contents:
1. v. 1. The structures of everyday life : the limits of the possible
2. v. 2. The wheels of commerce
3. v. 3. The perspective of the world.


In [10]:
# CHECKPOINT — after 5/7 and 6/7: the original table is still separate.
print("After TOC flattening")
print(f"• source_books is unchanged: {len(source_books):,} books × {len(source_books.columns):,} columns.")
print(f"• toc_items is a separate working view: {len(toc_items):,} TOC-entry rows for {toc_items['parent_asin'].nunique():,} books.")
print("• Next we audit TOC entries, clean their text, then join one cleaned toc_text column back to the full book table.")
print(f"Flattened TOC — same real book: {example_parent_asin}")
display(
    toc_items.loc[toc_items["parent_asin"] == example_parent_asin,
                  ["parent_asin", "toc_sequence", "toc_text_raw"]]
)

After TOC flattening
• source_books is unchanged: 9,034 books × 38 columns.
• toc_items is a separate working view: 141,495 TOC-entry rows for 9,034 books.
• Next we audit TOC entries, clean their text, then join one cleaned toc_text column back to the full book table.
Flattened TOC — same real book: 000216132X


,parent_asin,toc_sequence,toc_text_raw
0,000216132X,1,v. 1. The structures of everyday life : the li...
1,000216132X,2,v. 2. The wheels of commerce
2,000216132X,3,v. 3. The perspective of the world.


In [11]:
# Inspect three full source records. random_state makes the sample repeatable.
sampled_books = source_books.sample(n=3, random_state=RANDOM_STATE)
for _, record in sampled_books.iterrows():
    print(json.dumps(record.to_dict(), indent=2, ensure_ascii=False, default=str))
    print("\n" + "─" * 100 + "\n")

{
  "amazon_business_money_record_number": "26513",
  "amazon_isbn_10": "0631228306",
  "amazon_isbn_13": "9780631228301",
  "amazon_raw_record_json": "{\"main_category\": \"Books\", \"title\": \"Carbonell Museum Studies: An Anthology of Contexts\", \"subtitle\": \"1st Edition\", \"author\": null, \"average_rating\": 4.6, \"rating_number\": 8, \"features\": [\"Museum Studies: An Anthology of Contexts\", \"provides a comprehensive interdisciplinary collection of approaches to museums and their relation to history, culture, philosophy and their adoring or combative publics.\", \"Brings together for the first time a wide array of texts that mix contemporary analysis with historical documentation\", \"Brings together for the first time a wide array of texts that mix contemporary analysis with historical documentation\", \"Includes five sections that highlight central themes in museum studies: issue-oriented contexts in museology; states of \\\"nature\\\"; the status of nations; history, me

## 1. Audit duplicates and missing values

Do not remove a book merely because it shares an ISBN or an Open Library edition with another Amazon record. First inspect the fields below; only exact duplicate `parent_asin`/sequence pairs or blank TOC text would be invalid at this grain.

In [12]:
book_counts = toc_items.groupby("parent_asin").agg(
    observed_entries=("toc_sequence", "size"),
    declared_entries=("toc_entry_count", "first"),
)

quality = pd.Series(
    {
        "TOC-entry rows": len(toc_items),
        "books (parent_asin)": toc_items["parent_asin"].nunique(),
        "duplicate parent_asin / sequence pairs": toc_items.duplicated(["parent_asin", "toc_sequence"]).sum(),
        "blank TOC entries": toc_items["toc_text_raw"].fillna("").str.strip().eq("").sum(),
        "books below the 3-entry rule": (book_counts["declared_entries"] < 3).sum(),
        "books whose observed and declared entry counts differ": (book_counts["observed_entries"] != book_counts["declared_entries"]).sum(),
        "books where raw and normalized TOC counts differ": (
            toc_items.drop_duplicates("parent_asin")["raw_toc_item_count"]
            != toc_items.drop_duplicates("parent_asin")["toc_entry_count"]
        ).sum(),
    },
    name="count",
).to_frame()
display(quality)
display(toc_items.isna().mean().sort_values(ascending=False).rename("missing_share").to_frame())

,count
TOC-entry rows,141495
books (parent_asin),9034
duplicate parent_asin / sequence pairs,0
blank TOC entries,0
books below the 3-entry rule,0
books whose observed and declared entry counts differ,0
books where raw and normalized TOC counts differ,17


,missing_share
toc_page,0.954952
toc_level,0.073289
parent_asin,0.000000
raw_toc_item_count,0.000000
toc_entry_count,0.000000
toc_sequence,0.000000
toc_text_raw,0.000000


## 2. Create book-level TOC text (without changing content)

No content transformation is applied yet: TOC numbers, punctuation, capitalization, page numbers, and hierarchy may be meaningful. We validate the raw entries and add one convenient book-level `toc_text` column while keeping every raw field available.

In [13]:
# 2.1 — Validate the raw TOC structure before using it.
duplicate_pairs = toc_items.duplicated(["parent_asin", "toc_sequence"]).sum()
blank_raw_entries = toc_items["toc_text_raw"].isna().sum() + toc_items["toc_text_raw"].eq("").sum()
assert duplicate_pairs == 0
assert blank_raw_entries == 0
print(f"2.1 — Validation passed: {duplicate_pairs} duplicate book/sequence pairs; {blank_raw_entries} blank raw entries.")

2.1 — Validation passed: 0 duplicate book/sequence pairs; 0 blank raw entries.


In [14]:
# 2.2 — Inspect a raw book, then reassemble its ordered raw entries into toc_text.
raw_book = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("Raw source record (before any content transformation):")
print(f"Title: {raw_book['title']}")
print(f"Amazon ISBN-10: {raw_book['amazon_isbn_10']}")
print(f"Amazon ISBN-13: {raw_book['amazon_isbn_13']}")
print(f"Canonical ISBN-13: {raw_book['canonical_isbn_13']}")
print(f"Parent ASIN: {raw_book['parent_asin']}")
print("\nOriginal nested toc_entries:")
print(json.dumps(raw_book["toc_entries"], indent=2, ensure_ascii=False))

toc_text_by_book = (
    toc_items.sort_values(["parent_asin", "toc_sequence"])
    .groupby("parent_asin")["toc_text_raw"]
    .agg("\n".join)
)
toc_text_by_book.name = "toc_text"
print(f"2.2 — Created one ordered TOC text value for {len(toc_text_by_book):,} books.")
print(f"\nTOC for {BOOK_TO_INSPECT}:\n{toc_text_by_book.loc[BOOK_TO_INSPECT]}")

Raw source record (before any content transformation):
Title: The wheels of commerce, vol.2: civilisation and capitalism 15th-18th
Amazon ISBN-10: 000216132X
Amazon ISBN-13: 9780002161329
Canonical ISBN-13: 9780002161329
Parent ASIN: 000216132X

Original nested toc_entries:
[
  {
    "level": null,
    "page": null,
    "sequence": "1",
    "source_key": "value",
    "text": "v. 1. The structures of everyday life : the limits of the possible"
  },
  {
    "level": null,
    "page": null,
    "sequence": "2",
    "source_key": "value",
    "text": "v. 2. The wheels of commerce"
  },
  {
    "level": null,
    "page": null,
    "sequence": "3",
    "source_key": "value",
    "text": "v. 3. The perspective of the world."
  }
]
2.2 — Created one ordered TOC text value for 9,034 books.

TOC for 000216132X:
v. 1. The structures of everyday life : the limits of the possible
v. 2. The wheels of commerce
v. 3. The perspective of the world.


## 3. strip(), lower case (when embedding words upcase/lowercase matter)


In [15]:
# 3 — Optional variant: trim only the outside whitespace and lowercase TOC text.
# The unchanged books['toc_text'] remains our baseline and is never overwritten.
books = source_books.copy().join(toc_text_by_book, on="parent_asin")
books["toc_text_lower"] = books["toc_text"].str.strip().str.lower()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("3 — Optional lowercase variant created; baseline toc_text is unchanged.")
print(f"Baseline:  {example['toc_text']!r}")
print(f"Lowercase: {example['toc_text_lower']!r}")

3 — Optional lowercase variant created; baseline toc_text is unchanged.
Baseline:  'v. 1. The structures of everyday life : the limits of the possible\nv. 2. The wheels of commerce\nv. 3. The perspective of the world.'
Lowercase: 'v. 1. the structures of everyday life : the limits of the possible\nv. 2. the wheels of commerce\nv. 3. the perspective of the world.'


## 4. dealing with numbers, punctuation, and symbols

In [16]:
# 4 — Optional variant: remove digits, punctuation, and symbols from a copy.
# This is deliberately not applied to toc_text because TOC markers such as 1. and Part II may matter.
import re
books["toc_text_letters_only"] = (
    books["toc_text"].str.replace(r"[^\w\s]|\d", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("4 — Optional letters-only variant created; baseline toc_text is unchanged.")
print(f"Baseline:     {example['toc_text']!r}")
print(f"Letters only: {example['toc_text_letters_only']!r}")

4 — Optional letters-only variant created; baseline toc_text is unchanged.
Baseline:     'v. 1. The structures of everyday life : the limits of the possible\nv. 2. The wheels of commerce\nv. 3. The perspective of the world.'
Letters only: 'v The structures of everyday life the limits of the possible v The wheels of commerce v The perspective of the world'


## 5. splitting

In [17]:
# 5 — Optional structural view: split the multiline TOC into its original entries.
# This creates a list for analysis; it does not change toc_text.
# pandas has no .str.splitlines(); our book-level toc_text uses newline separators.
books["toc_lines"] = books["toc_text"].str.split("\n")
books["toc_line_count"] = books["toc_lines"].str.len()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"5 — Split TOC into {example['toc_line_count']} lines for {BOOK_TO_INSPECT}.")
for position, line in enumerate(example["toc_lines"], start=1):
    print(f"{position}. {line}")

5 — Split TOC into 3 lines for 000216132X.
1. v. 1. The structures of everyday life : the limits of the possible
2. v. 2. The wheels of commerce
3. v. 3. The perspective of the world.


## 6. tokenizing

In [18]:
# 6 — Optional token view: extract word-like tokens into a separate list.
# Most vectorizers/tokenizers do this themselves, so toc_text stays the baseline input.
books["toc_tokens"] = books["toc_text"].str.findall(r"\b\w+\b")
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"6 — Extracted {len(example['toc_tokens'])} tokens for {BOOK_TO_INSPECT}.")
print(example["toc_tokens"][:40])

6 — Extracted 25 tokens for 000216132X.
['v', '1', 'The', 'structures', 'of', 'everyday', 'life', 'the', 'limits', 'of', 'the', 'possible', 'v', '2', 'The', 'wheels', 'of', 'commerce', 'v', '3', 'The', 'perspective', 'of', 'the', 'world']


## 7. removing "stopwords"

In [19]:
# 7 — Optional token variant: remove English stopwords from a copy of the token list.
# Keep this experimental: words such as 'part' or 'chapter' may matter for TOCs.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
books["toc_tokens_no_stopwords"] = books["toc_tokens"].map(
    lambda tokens: [token for token in tokens if token.casefold() not in ENGLISH_STOP_WORDS]
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"7 — Tokens: {len(example['toc_tokens'])}; after stopword removal: {len(example['toc_tokens_no_stopwords'])}.")
print(example["toc_tokens_no_stopwords"][:40])

7 — Tokens: 25; after stopword removal: 15.
['v', '1', 'structures', 'everyday', 'life', 'limits', 'possible', 'v', '2', 'wheels', 'commerce', 'v', '3', 'perspective', 'world']


## 8. lemmatizing

In [20]:
# 8 — Optional token variant: apply simple noun lemmatization to a copy.
# This requires nltk in the current notebook kernel; all earlier steps work without it.
try:
    from nltk.stem import WordNetLemmatizer
except ModuleNotFoundError:
    print("8 — Skipped: nltk is not installed in this notebook kernel.")
    print("Run `%pip install -e .` from the project folder, restart the kernel, then rerun this optional cell.")
else:
    lemmatizer = WordNetLemmatizer()
    try:
        books["toc_tokens_lemmatized"] = books["toc_tokens"].map(
            lambda tokens: [lemmatizer.lemmatize(token.casefold()) for token in tokens]
        )
    except LookupError:
        print("8 — Skipped: the NLTK WordNet data is not installed.")
        print("Run `import nltk; nltk.download('wordnet')`, then rerun this optional cell.")
    else:
        example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
        print("8 — Original tokens versus optional noun-lemmatized tokens:")
        print(f"Original:    {example['toc_tokens'][:40]}")
        print(f"Lemmatized:  {example['toc_tokens_lemmatized'][:40]}")

8 — Original tokens versus optional noun-lemmatized tokens:
Original:    ['v', '1', 'The', 'structures', 'of', 'everyday', 'life', 'the', 'limits', 'of', 'the', 'possible', 'v', '2', 'The', 'wheels', 'of', 'commerce', 'v', '3', 'The', 'perspective', 'of', 'the', 'world']
Lemmatized:  ['v', '1', 'the', 'structure', 'of', 'everyday', 'life', 'the', 'limit', 'of', 'the', 'possible', 'v', '2', 'the', 'wheel', 'of', 'commerce', 'v', '3', 'the', 'perspective', 'of', 'the', 'world']
